In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import statsmodels.formula.api as smf
import warnings; warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
def check_coln(df):
    coln_type = {
        "pheno": float,
        "age": int,
        "sex": int,
    }
    
    for coln, dtype in coln_type.items():
        assert coln in df.columns, f"No {coln} in dataframe."
        
        df[coln] = df[coln].astype(dtype)
    
    return df
            
def std_col(vals):
    return (vals - np.nanmean(vals)) / np.nanstd(vals)
    
def reg_out_cov(df):
    # validate colnames
    df = check_coln(df)
    
    # regress out covariate
    formula = f"pheno ~ 1 + sex + age + age^2 + sex*age + sex*age^2"
    ll = smf.ols(formula, data=df).fit()
    
    # standardize phenotype
    df["residual"] = std_col(ll.resid)
    return df

# UKB

# GS

In [3]:
pheno_fn = "/data/jerrylee/data/GS/phenotypes.xls"
output_folder = "/home/jerrylee/data/pjt/BIGFAM.v.0.1/data/GS/phenotype"

In [7]:
# load data
df_pheno = pd.read_csv(pheno_fn, sep='\t')
df_pheno.head(3)

,id,age,sex,brachial_L,pedis_L,tibial_L,brachial_R,pedis_R,tibial_R,leg_L,...,FEV_3,FEF_3,FEV,FEF,FVC_1,FVC_2,FVC_3,FVC,ratio,expected
0,90826,51,M,150.0,158.0,176.0,146.0,154.0,168.0,176.0,...,3.63,6.03,3.93,6.03,4.26,4.53,4.10,4.53,0.87,3.66
1,83887,49,M,130.0,140.0,124.0,136.0,144.0,140.0,140.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.84
2,32660,63,F,122.0,140.0,142.0,140.0,132.0,140.0,142.0,...,2.74,2.71,2.74,2.91,2.98,3.26,3.52,3.52,0.78,2.53


In [50]:
phenos = df_pheno.columns.drop(["id", "age", "sex"])
excludes = [
    "tname", "B", "BS", "F", "G", "M", "S", "Y", "L", "R", "id", "dtRecorded", "dtSystem", "1", "2", "3", "4", "A", "P", "No_Serial_Comparison", "Type1", "Serial", "race", "rcode", "brothers", "sisters", "type"
    ]

In [49]:
for pheno in tqdm(phenos):
    # filtering
    if pheno in excludes: continue
    if len(pheno.split("_")) > 1:
        if pheno.split("_")[1] in excludes:
            continue
        if pheno.split("_")[-1] in excludes:
            continue
    
    tmp = (df_pheno[["id", "age", "sex", pheno]]
           .copy()
           .rename(columns={pheno: "pheno", "id": "eid"})
    )
    
    # recode sex
    mapper = {"M": 0, "F": 1}
    tmp["sex"] = tmp["sex"].replace(mapper)
    
    # regress out
    df = reg_out_cov(tmp)
    
    # save 
    (df[["eid", "eid", "residual"]]
    .to_csv(
            f"{output_folder}/{pheno}.phen",
            sep='\t',
            index=False
        ))

100%|██████████| 233/233 [00:03<00:00, 64.36it/s] 


In [46]:
pheno

'race'

In [47]:
tmp

,eid,age,sex,pheno
0,90826,51,0,White
1,83887,49,0,White
2,32660,63,1,White
3,90831,60,1,White
4,68776,55,1,White
...,...,...,...,...
19977,46093,60,1,NaN
19978,71541,22,1,NaN
19979,97675,36,1,NaN
19980,36172,27,1,NaN


In [28]:
pheno.split("_")[1] in excludes

False

In [29]:
pheno.split("_")[1]

'B'

In [30]:
excludes

['tname']